# Física BR · Taller de aplicación: Física y Ciberseguridad I — Canales Mecánicos
### Física BR — Física Mecánica
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
*Física interactiva para ingeniería.* Taller de aplicación para cerrar el curso. Muchos ataques informáticos no rompen
el cifrado: aprovechan la física del aparato. Un sonido a la frecuencia justa detiene un disco duro, el giroscopio de
un celular capta la voz, un péndulo caótico produce claves impredecibles y el acelerómetro delata qué teléfono es.
Aquí analizarás cada caso con lo que ya sabes: rotación, oscilador forzado, movimiento armónico simple, dinámica y
segunda ley de Newton. Requiere **Rotación**, **Oscilaciones** y **Oscilaciones Amortiguadas**.

**Convenciones:** tres cifras significativas · coma decimal · g = 9,81 m/s² · los datos de equipos son didácticos
(sintéticos) salvo los que se citan de una fuente · se estudia la física del ataque y de la defensa, no cómo atacar.

**Ecuaciones clave:**
- Rotación:  ω = 2π·f ,  f = rpm/60 ,  v = ω·r ,  a_c = ω²·r
- Oscilador forzado:  A/A_estática = 1/√[(1 − r²)² + (2ζ·r)²] ,  r = f/f₀ ,  en resonancia A/A_estática = 1/(2ζ) = Q
- MAS y muestreo:  x(t) = A·cos(2π·f·t) ;  con f_m muestras por segundo solo se reconstruye hasta f_m/2 (Nyquist);
  una frecuencia mayor aparece «doblada» como f_alias = |f − k·f_m| (k entero que la deja entre 0 y f_m/2)
- Péndulo simple:  T = 2π·√(L/g)      ·      caos:  separación δ(t) ≈ δ₀·e^(λ·t)
- Acelerómetro (masa y resorte):  k·x = m·a  →  x = m·a/k ,  f₀ = (1/2π)·√(k/m)

**Fuentes de los casos:** C. Bolton et al., «Blue Note», *IEEE S&P* 2018 · Y. Michalevsky, D. Boneh y G. Nakibly,
«Gyrophone», *USENIX Security* 2014 · Cloudflare, LavaRand (2017) y sus péndulos dobles de Londres · J. Zhang,
A. Beresford e I. Sheret, «SensorID», *IEEE S&P* 2019.

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Competencias: [CL, UFT, UCM, PRC, ULC].

---
> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*


---
## Punto 1 — Un sonido que detiene un disco duro: rotación y resonancia (Ing. de Sistemas/Seguridad de hardware)  ★★★  [PRC, UCM]
Un disco duro gira a **7 200 rpm**; la pista que se escribe está a **r = 4,50 cm** del eje. El brazo que sostiene el
cabezal se modela como un oscilador con frecuencia natural **f₀ = 5,00 kHz** y amortiguamiento **ζ = 0,0250**
(datos didácticos). Un parlante cercano lo empuja como una fuerza periódica que, aplicada sin vibrar, lo desplazaría
**1,00 nm** (desplazamiento estático equivalente). Para escribir, el cabezal debe mantenerse a menos de **10,0 nm**
del centro de la pista (dato didáctico).

**Rotación:**  f = rpm/60 ,  ω = 2π·f ,  v = ω·r

**Oscilador forzado:**  A = A_estática/√[(1 − r²)² + (2ζ·r)²] ,  r = f_sonido/f₀

> 🔧 **Aplicación en ingeniería:** en 2018, investigadores de la Universidad de Michigan y de Zhejiang mostraron que un
> tono de 5 kHz hace perder rendimiento a un disco duro y, a mayor intensidad, detiene la escritura; con eso hicieron
> fallar un computador y un grabador de video de vigilancia (Bolton et al., *IEEE S&P* 2018). La defensa propuesta es
> un controlador que compense la vibración. La física de fondo es la del oscilador forzado: importa la frecuencia.

> 🤖 **TIP-IA — Verificador crítico:**
> Dile a un asistente: *"Calculé la amplitud de vibración del cabezal de un disco duro con la fórmula del oscilador forzado. Actúa como evaluador escéptico: pregúntame qué pasa si el sonido está un 10 % por debajo de la frecuencia natural y por qué un tono suave puede hacer más daño que uno fuerte. No me des la respuesta."* Primero resuelves tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',')

P1_f0, P1_z, P1_x0, P1_tol = 5000.0, 0.025, 1.0, 10.0

def _amp(f, f0=P1_f0, z=P1_z):
    r = f/f0; return 1/np.sqrt((1 - r**2)**2 + (2*z*r)**2)

def esquema():
    import matplotlib.patches as _mp
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    a1.add_patch(_mp.Circle((0, 0), 1.0, color='#bdc3c7')); a1.add_patch(_mp.Circle((0, 0), 0.15, color='#7f8c8d'))
    a1.add_patch(_mp.Circle((0, 0), 0.75, fill=False, ls='--', color='#2980b9')); a1.text(-0.35, -0.55, 'pista', fontsize=8, color='#2980b9')
    a1.plot([1.5, 0.75], [-0.9, 0.0], color='#2c3e50', lw=4); a1.scatter([0.75], [0.0], color='#c0392b', s=40, zorder=5); a1.text(1.0, 0.1, 'cabezal', fontsize=8)
    a1.annotate('', xy=(0.55, 0.62), xytext=(0.78, 0.28), arrowprops=dict(arrowstyle='-|>', color='#8e44ad')); a1.text(0.2, 0.75, 'ω', fontsize=12, color='#8e44ad')
    for k in range(3): a1.add_patch(_mp.Arc((1.9, 0.6), 0.3 + 0.25*k, 0.3 + 0.25*k, theta1=120, theta2=240, color='#e67e22'))
    a1.text(1.75, 1.05, 'sonido', fontsize=8, color='#e67e22'); a1.set_xlim(-1.2, 2.2); a1.set_ylim(-1.2, 1.2); a1.set_aspect('equal'); a1.axis('off')
    a1.set_title('Disco que gira y brazo que vibra', fontsize=10)
    f = np.linspace(500, 10000, 800); a2.plot(f/1000, P1_x0*_amp(f), color='#8e44ad', lw=2)
    a2.axhline(P1_tol, ls='--', color='#c0392b'); a2.text(0.6, P1_tol*1.05, 'tolerancia del cabezal', fontsize=8, color='#c0392b')
    a2.set_xlabel('frecuencia del sonido (kHz)'); a2.set_ylabel('amplitud del cabezal (nm)'); a2.set_title('Solo una banda estrecha es peligrosa', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def disco(f_kHz=5.0, x_est_nm=1.0, zeta=0.025):
    f = f_kHz*1000; A = x_est_nm*_amp(f, P1_f0, zeta)
    print_es(f"r = f/f₀ = {_c3(f/P1_f0)}   |   factor de amplificación = {_c3(_amp(f, P1_f0, zeta))}   |   amplitud del cabezal = {_c3(A)} nm   |   Q = 1/(2ζ) = {_c3(1/(2*zeta))}")
    print_es('✅ El cabezal se mantiene dentro de la tolerancia.' if A <= P1_tol else '⚠️ El cabezal sale de la pista: la escritura se detiene.')
    ff = np.linspace(500, 10000, 800)
    plt.figure(figsize=(8, 3.3)); plt.plot(ff/1000, x_est_nm*_amp(ff, P1_f0, zeta), color='#8e44ad', lw=2); plt.axhline(P1_tol, ls='--', color='#c0392b')
    plt.scatter([f_kHz], [A], color='#c0392b', zorder=5); plt.xlabel('frecuencia (kHz)'); plt.ylabel('amplitud (nm)'); plt.title('Respuesta del brazo del cabezal', fontsize=10)
    plt.tight_layout(); plt.show()

def solucion():
    f = 7200/60; w = 2*np.pi*f; r = 0.0450
    print_es(f"a) f = 7 200/60 = {_c3(f)} Hz ;  ω = 2π·f = {_c3(w)} rad/s ;  periodo = {_c3(1000/f)} ms")
    print_es(f"b) Velocidad de la pista bajo el cabezal: v = ω·r = {_c3(w*r)} m/s ;  aceleración centrípeta = ω²·r = {_sci(w**2*r)} m/s² ({_c3(w**2*r/9.81)} veces g)")
    print_es(f"c) En resonancia (5,00 kHz): A = 1,00 nm/(2·0,0250) = {_c3(P1_x0*_amp(5000))} nm > 10,0 nm → el cabezal sale de la pista.")
    print_es(f"d) A 4,50 kHz (10 % por debajo): factor {_c3(_amp(4500))} → A = {_c3(P1_x0*_amp(4500))} nm: dentro de la tolerancia.")
    print_es(f"e) Para llegar a 10,0 nm: en resonancia basta un empuje estático de {_c3(10/_amp(5000))} nm; a 10,0 kHz (factor {_c3(_amp(10000))}) haría falta {_c3(10/_amp(10000))} nm,")
    print_es(f"   es decir, {_c3(_amp(5000)/_amp(10000))} veces más presión de sonido: la frecuencia pesa más que el volumen.")
    from scipy.integrate import solve_ivp
    w0 = 2*np.pi*P1_f0; wf = 2*np.pi*5000
    sol = solve_ivp(lambda t, y: [y[1], -2*P1_z*w0*y[1] - w0**2*y[0] + w0**2*P1_x0*np.cos(wf*t)], (0, 0.02), [0, 0], max_step=2e-6, rtol=1e-8)
    print_es(f"Verificación (integración de x'' + 2ζω₀x' + ω₀²x = ω₀²·x_est·cos ωt hasta el régimen permanente): amplitud = {_c3(np.max(np.abs(sol.y[0][sol.t > 0.015])))} nm ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Frecuencia de rotación del disco (en Hz):",
    120, 0.02, "Hz", "f=7 200 rpm/60=120 Hz (ω=754 rad/s).", id_preg="CS1_p1_1", peso=2.0)
quiz_numerico("Amplitud de vibración del cabezal con un tono de 5,00 kHz (en nm):",
    20.0, 0.02, "nm", "En resonancia el factor es 1/(2ζ)=20,0; A=20,0·1,00 nm=20,0 nm.", id_preg="CS1_p1_2", peso=3.0)
quiz_numerico("Amplitud del cabezal si el tono es de 4,50 kHz, un 10 % por debajo (en nm):",
    5.12, 0.02, "nm", "r=0,900; factor=1/√(0,190²+0,0450²)=5,12; A=5,12 nm.", id_preg="CS1_p1_3", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: el técnico de soporte asegura 'para tumbar un disco con sonido hace falta un ruido ensordecedor; lo que importa es el volumen, no el tono'. ¿Correcto o solo convincente?",
    ['Solo convincente: en resonancia el empuje se amplifica 20,0 veces; a 10,0 kHz haría falta unas 60 veces más presión para el mismo efecto',
     'Correcto: la amplitud del cabezal es proporcional a la intensidad del sonido, sin importar la frecuencia',
     'Correcto: un disco sellado no puede vibrar con el sonido del aire',
     'Solo convincente: el tono importa, pero lo mejor es una frecuencia muy baja, cercana a la de giro'], 0,
    'Es cierto que más volumen empuja más, y eso lo hace convincente. Pero el brazo es un oscilador forzado: cerca de f₀ el factor llega a Q = 20,0, y lejos cae por debajo de 1. Bolton et al. usaron un tono en la banda de resonancia, no un ruido cualquiera.',
    id_preg='CS1_p1_concep_1', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(disco),
    f_kHz=FloatSlider(value=3.0,min=0.5,max=10,step=0.05,description='f (kHz)'),
    x_est_nm=FloatSlider(value=0.5,min=0.05,max=5,step=0.05,description='x_est (nm)'),
    zeta=FloatSlider(value=0.1,min=0.005,max=0.5,step=0.005,description='ζ'));

---
## Punto 2 — El giroscopio que escucha: MAS y muestreo (Ing. de Sistemas/Privacidad móvil)  ★★★  [CL, PRC]
El giroscopio de un celular es una masa que vibra sobre resortes microscópicos; el sonido también la sacude. Android
entregaba sus lecturas a **200 muestras por segundo** (Michalevsky et al., 2014). La voz de un hombre adulto tiene su
frecuencia fundamental entre **85 y 180 Hz**. Considera un tono de **150 Hz**: x(t) = A·cos(2π·150·t).

**Muestreo:**  se toman valores en t_n = n/f_m

**Nyquist:**  solo se reconstruye fielmente hasta f_m/2

**Alias:**  una frecuencia f > f_m/2 aparece como f_alias = |f − k·f_m| , con el entero k que deja el resultado entre 0 y f_m/2

> 🔧 **Aplicación en ingeniería:** el giroscopio no pide permiso al usuario, a diferencia del micrófono. Con 200
> muestras por segundo, los autores reconocieron el género del hablante y algunas palabras, y al combinar dos teléfonos
> mejoraron el resultado. La defensa es física y simple: limitar la frecuencia de muestreo (los navegadores la
> limitaban a 20 Hz) o filtrar las frecuencias altas.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pídele a un asistente: *"Quiero entender el aliasing dibujando un coseno de 150 Hz y marcando encima los puntos tomados 200 veces por segundo. Ayúdame a ver qué curva más lenta pasa por esos mismos puntos, haciéndome preguntas paso a paso, sin darme la respuesta."* Primero intenta tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',')

def _alias(f, fm):
    r = f % fm; return min(r, fm - r)

def esquema():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    t = np.linspace(0, 0.04, 2000); tn = np.arange(0, 0.0401, 1/200)
    a1.plot(t*1000, np.cos(2*np.pi*150*t), color='#bdc3c7', lw=1.2, label='tono real de 150 Hz')
    a1.plot(t*1000, np.cos(2*np.pi*50*t), '--', color='#c0392b', lw=1.5, label='lo que «parece»: 50 Hz')
    a1.scatter(tn*1000, np.cos(2*np.pi*150*tn), color='#2c3e50', zorder=5, s=20, label='muestras a 200 Hz')
    a1.set_xlabel('t (ms)'); a1.set_ylabel('x (u. a.)'); a1.legend(fontsize=7, loc='lower right'); a1.set_title('Pocas muestras: el tono se disfraza', fontsize=10)
    f = np.linspace(0, 400, 801); a2.plot(f, [_alias(x, 200) for x in f], color='#8e44ad', lw=2)
    a2.axvspan(85, 180, color='#f5cba7', alpha=0.5); a2.text(95, 92, 'voz masculina', fontsize=8)
    a2.set_xlabel('frecuencia real (Hz)'); a2.set_ylabel('frecuencia aparente (Hz)'); a2.set_title('La frecuencia aparente se «dobla» en 100 Hz', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def muestreo(f_tono=150.0, f_m=200.0):
    fa = _alias(f_tono, f_m)
    print_es(f"Nyquist = f_m/2 = {_c3(f_m/2)} Hz   |   frecuencia aparente = {_c3(fa)} Hz   |   muestras por ciclo del tono = {_c3(f_m/f_tono)}")
    print_es('✅ El tono se reconstruye fielmente.' if f_tono < f_m/2 else '⚠️ El tono aparece disfrazado (alias): se filtra información, pero deformada.')
    t = np.linspace(0, 10/f_m, 2000); tn = np.arange(0, 10/f_m + 1e-12, 1/f_m)
    plt.figure(figsize=(8, 3.3)); plt.plot(t*1000, np.cos(2*np.pi*f_tono*t), color='#bdc3c7'); plt.plot(t*1000, np.cos(2*np.pi*fa*t), '--', color='#c0392b')
    plt.scatter(tn*1000, np.cos(2*np.pi*f_tono*tn), color='#2c3e50', zorder=5, s=18); plt.xlabel('t (ms)'); plt.ylabel('x (u. a.)')
    plt.title('Tono real, muestras y curva aparente', fontsize=10); plt.tight_layout(); plt.show()

def solucion():
    print_es("a) Con 200 muestras por segundo, el intervalo entre muestras es 1/200 = 5,00 ms y el límite de Nyquist es 100 Hz.")
    print_es(f"b) El tono de 150 Hz está por encima: f_alias = |150 − 1·200| = {_c3(_alias(150, 200))} Hz. Las muestras coinciden con las de un coseno de 50,0 Hz.")
    print_es(f"c) Un tono de 120 Hz aparece en {_c3(_alias(120, 200))} Hz; uno de 85 Hz se ve tal cual. La voz no se pierde: se deforma.")
    print_es(f"d) Con el límite de 20 muestras por segundo de los navegadores: 150 Hz aparece en {_c3(_alias(150, 20))} Hz y el límite fiel es 10,0 Hz.")
    print_es("e) Con dos teléfonos que muestrean alternados (equivalente a 400 Hz), el límite sube a 200 Hz y 150 Hz se ve sin disfraz.")
    n = np.arange(400); a = np.cos(2*np.pi*150*n/200); b = np.cos(2*np.pi*50*n/200)
    F = np.abs(np.fft.rfft(a)); fr = np.fft.rfftfreq(400, 1/200)
    print_es(f"Verificación: las muestras de 150 Hz y de 50,0 Hz son idénticas (diferencia máxima {np.max(np.abs(a - b)):.1e}); la FFT de las muestras tiene su pico en {_c3(fr[np.argmax(F)])} Hz ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Frecuencia máxima que se reconstruye fielmente con 200 muestras por segundo (en Hz):",
    100, 0.02, "Hz", "Nyquist: f_m/2=200/2=100 Hz.", id_preg="CS1_p2_1", peso=2.0)
quiz_numerico("¿En qué frecuencia aparente se ve el tono de 150 Hz muestreado a 200 Hz? (en Hz)",
    50.0, 0.02, "Hz", "f_alias=|150−200|=50,0 Hz.", id_preg="CS1_p2_2", peso=3.0)
quiz_numerico("Si el navegador limita el muestreo a 20 muestras por segundo, ¿en qué frecuencia aparente se ve el tono de 150 Hz? (en Hz)",
    10.0, 0.02, "Hz", "150=7·20+10 → resto 10 Hz, que ya está en el límite de 10 Hz: f_alias=10,0 Hz.", id_preg="CS1_p2_3", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un desarrollador afirma 'el giroscopio mide 200 veces por segundo y la voz está entre 85 y 180 Hz, por encima del límite de 100 Hz; el ataque es imposible'. ¿Correcto o solo convincente?",
    ['Solo convincente: los tonos por encima de 100 Hz no desaparecen, se doblan (150 Hz aparece en 50 Hz); la información queda, deformada',
     'Correcto: lo que está por encima de f_m/2 se elimina por completo de las muestras',
     'Correcto: un giroscopio mide rotación, y el sonido no produce ninguna fuerza sobre él',
     'Solo convincente: el giroscopio capta la voz completa, porque su masa vibra a 20 kHz'], 0,
    'Que el límite fiel es 100 Hz es cierto, y eso lo hace convincente. Pero muestrear no filtra: una frecuencia mayor deja muestras idénticas a las de otra más baja. Por eso la defensa es limitar el muestreo y filtrar antes de muestrear.',
    id_preg='CS1_p2_concep_2', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(muestreo),
    f_tono=FloatSlider(value=230.0,min=5,max=500,step=1,description='f tono (Hz)'),
    f_m=FloatSlider(value=100.0,min=10,max=1000,step=5,description='f muestreo (Hz)'));

---
## Punto 3 — Números impredecibles desde un péndulo doble: dinámica y caos (Ing. de Sistemas/Criptografía)  ★★★  [UCM, PRC]
Las claves criptográficas necesitan números que nadie pueda predecir. Cloudflare filma una pared de lámparas de lava
en San Francisco y, en su oficina de Londres, péndulos dobles caóticos, y usa esas imágenes como fuente adicional de
aleatoriedad. Considera un péndulo doble con dos brazos de **L = 0,500 m** y dos masas de **1,00 kg**, soltado desde
θ₁ = 120° y θ₂ = −10,0°. En la simulación de este taller, dos lanzamientos que difieren en δ₀ en el ángulo inicial se
separan **0,100 rad** a los **8,28 s** si δ₀ = **1,00×10⁻⁶ rad**, y a los **13,4 s** si δ₀ = **1,00×10⁻⁹ rad**.

**Péndulo simple (un brazo):**  T = 2π·√(L/g)

**Crecimiento de la separación:**  δ(t) ≈ δ₀·e^(λ·t)

**Tiempo de predicción:**  t = ln(Δ/δ₀)/λ   (Δ = separación tolerada)

> 🔧 **Aplicación en ingeniería:** un computador solo produce números pseudoaleatorios: si se conoce el estado
> inicial, se repiten. Un sistema físico caótico obedece leyes deterministas, pero amplifica cualquier diferencia
> mínima en el estado inicial, y ninguna medición es infinitamente precisa. Cloudflare mezcla esa entropía física con
> otras fuentes; no la usa sola.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que un péndulo doble no puede ser fuente de aleatoriedad, porque obedece las leyes de Newton y es determinista. Dame el mejor contraargumento y pregúntame qué precisión de medición haría falta para predecirlo un minuto. No me des números."* Primero razona tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',')

G9, P3_L = 9.81, 0.500

def _dp(t, y, L=P3_L):
    t1, t2, w1, w2 = y; d = t1 - t2; den = 3 - np.cos(2*d)
    a1 = (-3*G9*np.sin(t1) - G9*np.sin(t1 - 2*t2) - 2*np.sin(d)*(w2**2*L + w1**2*L*np.cos(d)))/(L*den)
    a2 = (2*np.sin(d)*(2*w1**2*L + 2*G9*np.cos(t1) + w2**2*L*np.cos(d)))/(L*den)
    return [w1, w2, a1, a2]

def _sim(th1, th2, d0, T=20.0, n=2001):
    from scipy.integrate import solve_ivp
    tt = np.linspace(0, T, n); y0 = [np.radians(th1), np.radians(th2), 0, 0]
    a = solve_ivp(_dp, (0, T), y0, t_eval=tt, method='DOP853', rtol=1e-11, atol=1e-12)
    b = solve_ivp(_dp, (0, T), [y0[0] + d0] + y0[1:], t_eval=tt, method='DOP853', rtol=1e-11, atol=1e-12)
    return tt, a, b

def esquema():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    t1, t2 = np.radians(120), np.radians(-10)
    x1, y1 = np.sin(t1), -np.cos(t1); x2, y2 = x1 + np.sin(t2), y1 - np.cos(t2)
    a1.plot([0, x1, x2], [0, y1, y2], color='#2c3e50', lw=2); a1.scatter([x1, x2], [y1, y2], s=80, color='#c0392b', zorder=5)
    a1.scatter([0], [0], color='#2c3e50', marker='s'); a1.text(x1 + 0.1, y1, 'θ₁ = 120°', fontsize=8); a1.text(x2 + 0.1, y2, 'θ₂ = −10,0°', fontsize=8)
    a1.set_xlim(-1.5, 2.5); a1.set_ylim(-1.5, 2.2); a1.set_aspect('equal'); a1.axis('off'); a1.set_title('Péndulo doble (dos brazos iguales)', fontsize=10)
    d0 = np.logspace(-15, -3, 50); a2.semilogx(d0, np.log(0.1/d0)/1.35, color='#8e44ad', lw=2)
    a2.set_xlabel('precisión de la medición inicial δ₀ (rad, log)'); a2.set_ylabel('tiempo de predicción (s)')
    a2.set_title('Mil veces más precisión: solo unos segundos más', fontsize=10); plt.tight_layout(); plt.show()
esquema()

def pendulo(th1=120.0, th2=-10.0, log10_d0=-6.0):
    if abs(th1) < 1e-9 and abs(th2) < 1e-9:
        print_es("⛔ BLOQUEADO: en reposo abajo no hay movimiento que separar."); return
    tt, a, b = _sim(th1, th2, 10**log10_d0, T=15.0, n=1501)
    sep = np.hypot(a.y[0] - b.y[0], a.y[1] - b.y[1]); i = np.argmax(sep > 0.1)
    ts = tt[i] if sep[i] > 0.1 else None
    print_es(f"δ₀ = {_sci(10**log10_d0)} rad   |   " + (f"las trayectorias se separan 0,100 rad a los {_c3(ts)} s" if ts else "en 15,0 s no se separan 0,100 rad (movimiento casi regular)"))
    fig, ax = plt.subplots(figsize=(8, 3.3)); ax.semilogy(tt, np.maximum(sep, 1e-16), color='#8e44ad', lw=1.5); ax.axhline(0.1, ls='--', color='#c0392b')
    ax.set_xlabel('t (s)'); ax.set_ylabel('separación (rad, log)'); ax.set_title('Dos lanzamientos casi iguales', fontsize=10); plt.tight_layout(); plt.show()

def solucion():
    T = 2*np.pi*np.sqrt(P3_L/G9)
    print_es(f"a) Un solo brazo como péndulo simple: T = 2π·√(0,500/9,81) = {_c3(T)} s")
    lam = np.log(1e-6/1e-9)/(13.4 - 8.28)
    print_es(f"b) Mil veces más precisión (de 10⁻⁶ a 10⁻⁹ rad) dio 13,4 − 8,28 = {_c3(13.4 - 8.28)} s más: λ = ln(1 000)/{_c3(13.4 - 8.28)} = {_c3(lam)} s⁻¹")
    print_es(f"c) Con δ₀ = 1,00×10⁻¹² rad: t ≈ 13,4 + ln(1 000)/λ = {_c3(13.4 + np.log(1000)/lam)} s. Cada factor de mil en precisión compra solo 5,12 s más.")
    print_es(f"d) Para predecir 60,0 s haría falta δ₀ ≈ 0,100·e^(−λ·60,0) = {_sci(0.1*np.exp(-lam*60))} rad: imposible de medir.")
    tt, a, b = _sim(120, -10, 1e-9)
    sep = np.hypot(a.y[0] - b.y[0], a.y[1] - b.y[1])
    E = lambda y: -(2*G9*P3_L*np.cos(y[0]) + G9*P3_L*np.cos(y[1])) + 0.5*(P3_L*y[2])**2 + 0.5*((P3_L*y[2])**2 + (P3_L*y[3])**2 + 2*P3_L**2*y[2]*y[3]*np.cos(y[0] - y[1]))
    print_es(f"Verificación (simulación con DOP853): separación de 0,100 rad a los {_c3(tt[np.argmax(sep > 0.1)])} s ✓ ; energía por kg al inicio y al final: {_c3(E(a.y[:, 0]))} y {_c3(E(a.y[:, -1]))} J/kg ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Periodo de un solo brazo de 0,500 m oscilando como péndulo simple (en s):",
    1.42, 0.02, "s", "T=2π√(0,500/9,81)=1,42 s.", id_preg="CS1_p3_1", peso=2.0)
quiz_numerico("Ritmo de separación λ estimado con los dos lanzamientos de la simulación (en s⁻¹):",
    1.35, 0.03, "s⁻¹", "λ=ln(10⁻⁶/10⁻⁹)/(13,4−8,28)=6,91/5,12=1,35 s⁻¹.", id_preg="CS1_p3_2", peso=3.0)
quiz_numerico("Con el modelo exponencial, ¿hasta qué tiempo se podría predecir si δ₀ = 1,00×10⁻¹² rad? (en s)",
    18.5, 0.02, "s", "Otro factor de mil suma el mismo intervalo: 13,4+5,12=18,5 s.", id_preg="CS1_p3_3", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante afirma 'el péndulo doble obedece las leyes de Newton; con las ecuaciones y un buen computador se predice su posición dentro de un minuto, así que no sirve para generar claves'. ¿Correcto o solo convincente?",
    ['Solo convincente: es determinista, pero cada factor de mil en precisión da solo unos 5 s más; un minuto exigiría medir el ángulo con unos 10⁻³⁶ rad',
     'Correcto: un sistema determinista siempre se puede predecir con un computador suficientemente potente',
     'Correcto: el péndulo doble es periódico, con periodo 1,42 s',
     'Solo convincente: el péndulo doble no obedece las leyes de Newton, por eso es aleatorio'], 0,
    'Que obedece las leyes de Newton es cierto, y eso lo hace convincente. Pero predecir exige conocer el estado inicial, y el caos amplifica cualquier error de forma exponencial. Ningún instrumento mide con 10⁻³⁶ rad: el límite es la medición, no el computador.',
    id_preg='CS1_p3_concep_3', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(pendulo),
    th1=FloatSlider(value=90.0,min=-170,max=170,step=1,description='θ₁ (°)'),
    th2=FloatSlider(value=30.0,min=-170,max=170,step=1,description='θ₂ (°)'),
    log10_d0=FloatSlider(value=-8.0,min=-14,max=-2,step=0.5,description='log₁₀ δ₀'));

---
## Punto 4 — El acelerómetro que delata al teléfono: masa, resorte y calibración (Ing. de Sistemas/Privacidad)  ★★★  [UCM, ULC]
El acelerómetro de un celular es una masa de prueba de **m = 1,00×10⁻⁹ kg** sujeta por resortes de constante
equivalente **k = 1,00 N/m** (datos didácticos). Con el teléfono quieto sobre la mesa, el chip convierte el
desplazamiento en un número entero A (cuentas) y la salida es **O = G·A + B**, donde G es la ganancia que se ajustó
en la fábrica para ese aparato. La ganancia nominal es **G₀ = 2,395×10⁻³ (m/s²)/cuenta**. Una página web lee
diferencias entre salidas consecutivas: **0,004814 ; −0,002407 ; 0,007221 m/s²**.

**Segunda ley en la masa de prueba:**  k·x = m·a  →  x = m·a/k ,  f₀ = (1/2π)·√(k/m)

**Calibración:**  O = G·A + B  →  ΔO = G·ΔA , con ΔA entero: G es el «paso» común de todas las diferencias

> 🔧 **Aplicación en ingeniería:** Zhang, Beresford y Sheret (*IEEE S&P* 2019) mostraron que la ganancia de calibración
> de cada sensor es casi única: con menos de 100 muestras y en menos de un segundo, una página o una app identifica el
> teléfono sin pedir permisos. Apple respondió agregando ruido a las salidas (iOS 12.2) y Google redondeándolas a
> múltiplos de la ganancia nominal (Android 11). La precisión de fábrica, irónicamente, se volvió una huella.

> 🤖 **TIP-IA — Generador de problemas:**
> Pídele a un asistente: *"Créame una variante: un magnetómetro o un giroscopio con otra ganancia nominal y una lista de diferencias entre lecturas. Pídeme hallar la ganancia real, su desviación porcentual y explicar por qué redondear las salidas a múltiplos de la ganancia nominal borra la huella. No me des la solución hasta que yo la intente."* Primero resuelves tú, después verificas.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',')

P4_m, P4_k, P4_G0, P4_G = 1.00e-9, 1.00, 2.395e-3, 2.407e-3
P4_dO = np.array([0.004814, -0.002407, 0.007221])

def esquema():
    import matplotlib.patches as _mp
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    a1.add_patch(_mp.Rectangle((0, 0), 3, 2.2, fill=False, color='#2c3e50', lw=2)); a1.add_patch(_mp.Rectangle((1.2, 0.7), 0.6, 0.6, color='#2980b9'))
    zz = np.linspace(1.3, 2.2, 60); a1.plot(1.5 + 0.08*np.sin((zz - 1.3)*40), zz, color='#7f8c8d'); zz = np.linspace(0, 0.7, 60); a1.plot(1.5 + 0.08*np.sin(zz*40), zz, color='#7f8c8d')
    a1.annotate('', xy=(2.3, 0.55), xytext=(2.3, 1.45), arrowprops=dict(arrowstyle='-|>', color='#c0392b')); a1.text(2.4, 0.95, 'g', color='#c0392b', fontsize=11)
    a1.text(0.15, 1.0, 'masa de\nprueba', fontsize=8); a1.set_xlim(-0.2, 3.3); a1.set_ylim(-0.2, 2.4); a1.set_aspect('equal'); a1.axis('off'); a1.set_title('Acelerómetro: masa y resorte', fontsize=10)
    A = np.arange(4090, 4100); a2.step(A, P4_G*A, where='mid', color='#8e44ad', label='este teléfono (G)'); a2.step(A, P4_G0*A + 0.035, where='mid', color='#27ae60', ls='--', label='nominal (G₀), desplazada')
    a2.set_xlabel('salida del convertidor A (cuentas)'); a2.set_ylabel('aceleración reportada (m/s²)'); a2.legend(fontsize=8); a2.set_title('Cada aparato tiene su propio «escalón»', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def sensor(m_ug=1.0, k=1.0, desvio_pct=0.5):
    m = m_ug*1e-9; G = P4_G0*(1 + desvio_pct/100)
    print_es(f"x en reposo = m·g/k = {_c3(m*9.81/k*1e9)} nm   |   f₀ = {_c3(np.sqrt(k/m)/(2*np.pi))} Hz   |   escalón G = {_sci(G)} (m/s²)/cuenta   |   cuentas para 9,81 m/s² = {9.81/G:.0f}")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.3)); A = np.arange(0, 12)
    a1.step(A, G*A*1000, where='mid', color='#8e44ad', label='este aparato'); a1.step(A, P4_G0*A*1000, where='mid', color='#27ae60', ls='--', label='nominal')
    a1.set_xlabel('cuentas'); a1.set_ylabel('salida (mm/s²)'); a1.legend(fontsize=8); a1.set_title('Escalones de salida', fontsize=10)
    aa = np.linspace(0, 2*9.81, 50); a2.plot(aa, m*aa/k*1e9, color='#2980b9', lw=2); a2.scatter([9.81], [m*9.81/k*1e9], color='#c0392b', zorder=5)
    a2.set_xlabel('aceleración (m/s²)'); a2.set_ylabel('desplazamiento de la masa (nm)'); a2.set_title('x = m·a/k', fontsize=10)
    plt.tight_layout(); plt.show()

def solucion():
    x = P4_m*9.81/P4_k; f0 = np.sqrt(P4_k/P4_m)/(2*np.pi)
    print_es(f"a) En reposo: x = m·g/k = 1,00×10⁻⁹·9,81/1,00 = {_sci(x)} m = {_c3(x*1e9)} nm ;  f₀ = {_c3(f0)} Hz")
    print_es("b) Cada diferencia es G por un entero: 0,004814 = 2·G ; −0,002407 = −1·G ; 0,007221 = 3·G")
    print_es("   → G = 0,002407 (m/s²)/cuenta = 2,407×10⁻³ (se conservan las cifras del dato: la huella está en la cuarta)")
    print_es(f"c) Desviación respecto al nominal: (2,407 − 2,395)/2,395 = {_c3(100*(P4_G/P4_G0 - 1))} %: pequeña, pero característica del aparato.")
    print_es(f"d) Con el teléfono quieto, la salida de 9,81 m/s² corresponde a unas {9.81/P4_G:.0f} cuentas; en otro teléfono a otras. Las lecturas coinciden;")
    print_es("   lo que delata es el tamaño del escalón. Redondear a múltiplos de G₀ o agregar ruido borra esa huella.")
    cand = np.linspace(2.0e-3, 2.8e-3, 80001)
    err = [np.max(np.abs(P4_dO/g - np.round(P4_dO/g))) for g in cand]
    ok = cand[np.array(err) < 1e-4]
    print_es(f"Verificación (búsqueda del mayor paso que hace enteras a las tres diferencias): G = {ok.max()*1e3:.3f}×10⁻³ (m/s²)/cuenta ✓".replace('.', ','))
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Desplazamiento de la masa de prueba con el teléfono quieto, x = m·g/k (en nm):",
    9.81, 0.02, "nm", "x=1,00×10⁻⁹·9,81/1,00=9,81×10⁻⁹ m=9,81 nm.", id_preg="CS1_p4_1", peso=2.0)
quiz_numerico("Ganancia G de este acelerómetro según las diferencias leídas (en milésimas de m/s² por cuenta, p. ej. 2,40):",
    2.407, 0.002, "×10⁻³ (m/s²)/cuenta", "Las diferencias son 2, −1 y 3 veces 0,002407: G=2,407×10⁻³.", id_preg="CS1_p4_2", peso=3.0)
quiz_numerico("Desviación porcentual de la ganancia respecto a la nominal (en %):",
    0.501, 0.03, "%", "(2,407−2,395)/2,395=0,00501=0,501 %.", id_preg="CS1_p4_3", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un usuario dice 'mi acelerómetro está bien calibrado y en reposo mide 9,81 m/s², igual que cualquier otro teléfono; no puede servir para identificarme'. ¿Correcto o solo convincente?",
    ['Solo convincente: la lectura coincide, pero el escalón G con que cambia es propio del aparato (0,501 % sobre el nominal) y se recupera de las diferencias',
     'Correcto: si todos los teléfonos marcan 9,81 m/s², sus datos son indistinguibles',
     'Correcto: la gravedad es igual para todos, así que el sensor no guarda información del aparato',
     'Solo convincente: lo que identifica al teléfono es la masa de prueba, que pesa distinto en cada uno'], 0,
    'Que la lectura en reposo es la misma es cierto, y eso lo hace convincente. Pero la huella no está en el valor sino en cómo cambia: las diferencias son múltiplos enteros de la ganancia de fábrica. Por eso la defensa fue redondear o agregar ruido, no recalibrar mejor.',
    id_preg='CS1_p4_concep_4', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(sensor),
    m_ug=FloatSlider(value=2.0,min=0.1,max=10,step=0.1,description='m (µg)'),
    k=FloatSlider(value=3.0,min=0.1,max=10,step=0.1,description='k (N/m)'),
    desvio_pct=FloatSlider(value=-1.0,min=-3,max=3,step=0.05,description='desvío G (%)'));

---
## ✅ Fin del taller Física y Ciberseguridad I — Canales Mecánicos (Física BR)
Ninguno de estos ataques rompió una clave: aprovecharon una resonancia, un muestreo escaso, la sensibilidad del caos
o un ajuste de fábrica. La física que explica el ataque también diseña la defensa. Sigue con **Física y
Ciberseguridad II** en Electricidad y Magnetismo. Experimenta con los sliders y **predice** antes de pulsar
"Mostrar solución". *Física interactiva para ingeniería — 2026-2.*


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# Calificación formativa del taller
calificacion_final(total_preguntas=16)
